- 전반적인 그림: https://python.langchain.com/v0.2/docs/concepts/#retrieval
- 과정: Source > Load > Transform > Embed > Store > Retrieve


## 6.1 Data Loaders and Splitters
> Load > Transform steps

- 데이터를 로딩하고, 텍스트를 여러 단위로 나누는 작업을 LangChain lib 에서 제공해주고 있음


In [1]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
character_text_splitter = CharacterTextSplitter(
    chunk_size=600, 
    chunk_overlap=100,
    separator="\n",
)
loader = TextLoader('files/1984_chapter_3_to_6.txt')

docs = loader.load_and_split(text_splitter=character_text_splitter)
len(docs)

141

In [2]:
docs[0:1]

[Document(metadata={'source': 'files/1984_chapter_3_to_6.txt'}, page_content="Chapter 3\n'There are three stages in your reintegration,' said O'Brien. 'There is\nlearning, there is understanding, and there is acceptance. It is time for\nyou to enter upon the second stage.'\nAs always, Winston was lying flat on his back. But of late his bonds were\nlooser. They still held him to the bed, but he could move his knees a\nlittle and could turn his head from side to side and raise his arms from\nthe elbow. The dial, also, had grown to be less of a terror. He could\nevade its pangs if he was quick-witted enough: it was chiefly when he")]

## 6.2 Tiktoken

- token 은 llm 이 이해하는 문자 단위임, characters length 가 아님
- https://platform.openai.com/tokenizer
- 한국어는 자모음 분리되어서 토크나이징 되는 것도 볼 수 있음
- https://github.com/openai/tiktoken?tab=readme-ov-file#-tiktoken
- 모델 이름으로도 찾을 수 있음
- https://python.langchain.com/v0.2/docs/how_to/split_by_token/#tiktoken

In [3]:
character_text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-4o-mini",
    chunk_size=600, 
    chunk_overlap=100,
    separator="\n",
)
loader = TextLoader('files/1984_chapter_3_to_6.txt')

docs = loader.load_and_split(text_splitter=character_text_splitter)
len(docs)

34

In [4]:
docs[0:1]

[Document(metadata={'source': 'files/1984_chapter_3_to_6.txt'}, page_content='Chapter 3\n\'There are three stages in your reintegration,\' said O\'Brien. \'There is\nlearning, there is understanding, and there is acceptance. It is time for\nyou to enter upon the second stage.\'\nAs always, Winston was lying flat on his back. But of late his bonds were\nlooser. They still held him to the bed, but he could move his knees a\nlittle and could turn his head from side to side and raise his arms from\nthe elbow. The dial, also, had grown to be less of a terror. He could\nevade its pangs if he was quick-witted enough: it was chiefly when he\nshowed stupidity that O\'Brien pulled the lever. Sometimes they got through\na whole session without use of the dial. He could not remember how many\nsessions there had been. The whole process seemed to stretch out over a\nlong, indefinite time--weeks, possibly--and the intervals between the\nsessions might sometimes have been days, sometimes only an hour 

## 6.3 Vectors (Embeddings)
> Embed Step

- 스포티파이 추천 방식
  - https://www.youtube.com/watch?v=2eWuYf-aZE4
- 현재 사용 가능한 embedding model
  -  https://platform.openai.com/docs/guides/embeddings/embedding-models

In [5]:
from config import langchain

langchain.setup_langchain_openai_envs()

In [6]:
from langchain_openai import OpenAIEmbeddings

embedder = OpenAIEmbeddings(model="text-embedding-3-small")

vector = embedder.embed_query("hi")
len(vector)

1536

## 6.4 Vector Store
> Store Step

- 여기선 Chroma local vector store 를 사용할 예정
- 파일을 이용한 embedding 캐싱을 적용할 예정

In [7]:
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter

character_text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-3.5-turbo",
    chunk_size=600, 
    chunk_overlap=100,
    separator="\n",
)
loader = TextLoader('files/1984_chapter_3_to_6.txt')
docs = loader.load_and_split(text_splitter=character_text_splitter)

openai_3_small_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
file_store = LocalFileStore('./.cache/')
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=openai_3_small_embeddings,
    document_embedding_cache=file_store
)

vectorstore = Chroma.from_documents(embedding=cached_embeddings, documents=docs)

In [8]:
results = vectorstore.similarity_search("where does winston live")

In [9]:
len(results)

4

In [10]:
results[0]

Document(metadata={'source': 'files/1984_chapter_3_to_6.txt'}, page_content="in human history--victory, victory, victory!'\nUnder the table Winston's feet made convulsive movements. He had not\nstirred from his seat, but in his mind he was running, swiftly running,\nhe was with the crowds outside, cheering himself deaf. He looked up again\nat the portrait of Big Brother. The colossus that bestrode the world!\nThe rock against which the hordes of Asia dashed themselves in vain! He\nthought how ten minutes ago--yes, only ten minutes--there had still been\nequivocation in his heart as he wondered whether the news from the front\nwould be of victory or defeat. Ah, it was more than a Eurasian army that\nhad perished! Much had changed in him since that first day in the Ministry\nof Love, but the final, indispensable, healing change had never happened,\nuntil this moment.\nThe voice from the telescreen was still pouring forth its tale of prisoners\nand booty and slaughter, but the shouting ou

## 6.6 RetrieveQA (Deprecated)

- chain_type
  - stuff: 검색된 document 를 system에 채워넣음
  - refine: 각 다큐먼트를 llm 에 피딩하면서 개선된 답변을 내놓을 수 있음
  - map reduce: 각 답변을 요약 > 각 요약본을 Llm 에 전달
  - map re-rank: 각 다큐먼트에 대해 답변을 생성하고 스코어를 매김 > 가장 좋은 스코어를 리턴할 수 있도록 함 

In [12]:
from langchain_openai import ChatOpenAI
from langchain.chains.retrieval_qa.base import RetrievalQA

llm = ChatOpenAI(model="gpt-3.5-turbo")
chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

In [13]:
chain.run("Describe Victory Mensions")

/Users/user/Library/Caches/pypoetry/virtualenvs/welcome-langchain-xP8O1TGX-py3.10/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:151: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  warn_deprecated(


'In the provided text, the concept of "Victory" is a significant theme, symbolizing the triumph of the ruling party (Big Brother) over its enemies. The word "Victory" is repeated multiple times in a celebratory manner, signaling a moment of success and achievement for the Party. It represents the ultimate control and power of the Party over its citizens, instilling a sense of loyalty and submission. Victory is portrayed as an all-encompassing force that brings about unity, obedience, and ultimately, love for Big Brother.'

In [14]:
llm = ChatOpenAI(model="gpt-3.5-turbo")
chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="refine",
    retriever=vectorstore.as_retriever()
)

In [15]:
chain.run("Describe the Winston character")

"The additional context provided delves deeper into Winston's character, showcasing his unwavering belief in the resilience of humanity and his defiance against the Party's oppressive regime. Despite O'Brien's attempts to break him and instill fear and submission, Winston remains steadfast in his belief that the Party's cruel and hateful methods will ultimately lead to its downfall.\n\nRefining the original answer with this context, we can further emphasize Winston's unwavering conviction in the ultimate triumph of life and humanity over the Party's oppressive rule. His refusal to accept O'Brien's vision of a world built on fear and hatred demonstrates his strong moral compass and his belief in the inherent goodness of humanity. This highlights Winston's rebellious spirit and his determination to resist the Party's attempts to control and manipulate him."

## 6.8 Stuff LCEL Chain

In [16]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.1)
retriever = vectorstore.as_retriever()
prompt = ChatPromptTemplate.from_messages([
    ('system', "You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up:\n\n{context}"),
    ('human', '{question}')
])
chain = {'context': retriever, 'question': RunnablePassthrough()} | prompt | llm

In [17]:
chain.invoke("Describe the Winston character")

AIMessage(content='Winston is a character who experiences a range of emotions, from pity for his ruined body to defiance against the Party. He shows resilience and a sense of moral superiority, even in the face of extreme adversity. Winston is intelligent and thoughtful, holding onto his beliefs despite the challenges he faces. He is depicted as someone who still holds onto love and loyalty, even in a society that seeks to break him down.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 2526, 'total_tokens': 2608}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-a2bff0a4-d1cb-4a44-a197-7392c6056e50-0', usage_metadata={'input_tokens': 2526, 'output_tokens': 82, 'total_tokens': 2608})

## 6.9 Map-Reduce LCEL Chain

In [18]:
from langchain_core.runnables import RunnableLambda

map_doc_prompt = ChatPromptTemplate.from_messages([
    ("system", """
Use the following portion of a long document to see if any of the text is relevant to answer the question. Return any relevant text verbatim. If there is no relevant text, return : ''
-------
{context}
"""),
    ("human", "{question}"),
])
map_doc_chain = map_doc_prompt | llm


def map_docs(inputs):
    documents = inputs["documents"]
    question = inputs["question"]
    return "\n\n".join(
        map_doc_chain.invoke(
            {"context": doc.page_content, "question": question}
        ).content
        for doc in documents
    )

map_chain = {
    "documents": retriever,
    "question": RunnablePassthrough(),
} | RunnableLambda(map_docs)

reduce_prompt = ChatPromptTemplate.from_messages([
    ('system', """
Given the following extracted parts of a long document and a question, create a final answer. 
If you don't know the answer, just say that you don't know. Don't try to make up an answer.
------
{context}
"""),
    ('human', '{question}'),
])

chain = {"context": map_chain, "question": RunnablePassthrough()} | reduce_prompt | llm

In [19]:
chain.invoke("Describe the Winston character")

AIMessage(content="Winston is a complex character who initially rebels against the oppressive Party in the dystopian society depicted in the text. He is portrayed as intelligent, introspective, and critical of the Party's control and manipulation. Despite his initial defiance, Winston is ultimately broken down through psychological and physical torture by O'Brien, a high-ranking Party member. Winston evolves from a defiant individual to one who is defeated and stripped of his beliefs and identity, becoming a symbol of resistance and the last vestige of humanity in a totalitarian world. He maintains a sense of loyalty and love towards Julia, showcasing resilience and inner strength, while also embodying hope and a belief in the resilience of the human spirit against oppressive forces.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 139, 'prompt_tokens': 593, 'total_tokens': 732}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': Non